# Segmentation Baseline Training (Phase 3)

Runs in Google Colab. Actual model training happens here, never on a
laptop. This notebook calls the repository's `evat.models` / `evat.training`
/ `evat.evaluation` modules — it does not reimplement the model or training
loop.

Dataset: YouTube-VOS (non-commercial research use only, see
`docs/datasets.md`). DAVIS is not used.

Strategy (do not skip ahead): SMOKE EXPERIMENT on a tiny subset first,
then a BASELINE EXPERIMENT on a larger-but-still-Colab-sized subset. Do
not attempt full-dataset training until the baseline is verified stable.

In [ ]:
%pip install -q -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os
from pathlib import Path

os.environ["EVAT_DATA_ROOT"] = "/content/data"  # example only; set to the real path
dataset_root = Path(os.environ["EVAT_DATA_ROOT"]) / "youtube_vos"

## SMOKE EXPERIMENT

Verify data loading, masks, forward/backward pass, checkpointing, and
evaluation on a tiny subset (a handful of videos) before scaling up.

In [ ]:
from torch.utils.data import DataLoader

from evat.data.datasets.youtube_vos import build_video_index
from evat.models.unet import UNet
from evat.training.config import SegmentationTrainingConfig
from evat.training.dataset import SegmentationDataset, collate_segmentation_batch
from evat.training.losses import BCEDiceLoss
from evat.training.trainer import Trainer

config = SegmentationTrainingConfig.from_yaml("configs/segmentation.yaml")
config.epochs = 1  # smoke experiment only

all_videos = build_video_index(dataset_root, split="train")
smoke_videos = all_videos[:4]  # tiny subset

smoke_dataset = SegmentationDataset(
    smoke_videos, dataset_root, height=config.input_height, width=config.input_width
)
smoke_loader = DataLoader(
    smoke_dataset, batch_size=config.batch_size, collate_fn=collate_segmentation_batch
)

model = UNet(
    in_channels=config.in_channels,
    out_channels=config.out_channels,
    base_channels=config.base_channels,
    depth=config.depth,
)
trainer = Trainer(model, BCEDiceLoss(), config, device=device)

smoke_history = trainer.fit(smoke_loader, smoke_loader)
print(smoke_history)

## BASELINE EXPERIMENT

Only run this after the smoke experiment above completes without errors.
Use a subset sized for free Colab (adjust based on actual GPU/session
limits observed).

In [ ]:
import time

config = SegmentationTrainingConfig.from_yaml("configs/segmentation.yaml")

val_videos = build_video_index(dataset_root, split="valid")

train_dataset = SegmentationDataset(
    all_videos,
    dataset_root,
    height=config.input_height,
    width=config.input_width,
    augment=True,
    seed=config.seed,
)
val_dataset = SegmentationDataset(
    val_videos, dataset_root, height=config.input_height, width=config.input_width
)
train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=collate_segmentation_batch
)
val_loader = DataLoader(
    val_dataset, batch_size=config.batch_size, collate_fn=collate_segmentation_batch
)

model = UNet(
    in_channels=config.in_channels,
    out_channels=config.out_channels,
    base_channels=config.base_channels,
    depth=config.depth,
)
trainer = Trainer(model, BCEDiceLoss(), config, device=device)

start = time.time()
history = trainer.fit(train_loader, val_loader)
runtime_seconds = time.time() - start
print(history)
print("runtime_seconds:", runtime_seconds)

In [ ]:
# Qualitative inspection: save a panel of frame | ground truth | prediction | overlay.
from evat.visualization.overlay import make_qualitative_panel

model.eval()
batch = next(iter(val_loader))
with torch.no_grad():
    logits = model(batch.image.to(device))
    preds = (torch.sigmoid(logits) >= 0.5).float().cpu()

panel = make_qualitative_panel(batch.image[0], batch.mask[0], preds[0])
panel.save("results/segmentation/qualitative_sample.png")
panel

In [ ]:
# Record the actual experiment — real numbers only, taken from the run above.
import subprocess

from evat.evaluation.experiment import save_experiment_record

git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
final_metrics = list(history.values())[-1]

save_experiment_record(
    "results/segmentation",
    "baseline_v1",
    config=config,
    metrics=final_metrics,
    git_commit=git_commit,
    dataset_version="<fill in verified YouTube-VOS version/date>",
    hardware=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    runtime_seconds=runtime_seconds,
)

## Save results

Update `docs/experiments.md` and `docs/architecture.md` with the actual
printed metrics/runtime from this run. Do not hand-edit numbers that were
not produced by this notebook.